# NERAL Helios v3 · Google Colab T4 · experimentelles 4-Bit-QLoRA

**Vier Schritte (erst ausführen, wenn der Datensatz auf GitHub verifiziert wurde):**

1. [colab.research.google.com](https://colab.research.google.com/) öffnen und dieses Notebook über GitHub oder als Datei laden.
2. **Laufzeit → Laufzeittyp ändern → T4 GPU** auswählen (die kostenlose T4 ist nicht garantiert).
3. Das geprüfte GitHub-Actions-Artefakt **neral-helios-v3-verified-dataset** als ZIP herunterladen und in der Upload-Zelle hochladen. Die ZIP muss train.jsonl, validation.jsonl, manifest.json und docker_proof.json enthalten.
4. **Laufzeit → Alle ausführen** starten, Upload bestätigen und nach dem Abschluss die Adapter-ZIP herunterladen. Später auf dem Heim-PC nach **D:** kopieren.

**Grenzen:** Die T4 hat 16 GB nominellen VRAM, unterstützt aber kein BF16. Hier wird explizit FP16 und 4-Bit-NF4 verwendet. Qwen3.5-9B-QLoRA auf der kostenlosen T4 ist **nicht garantiert lauffähig oder numerisch stabil**; Unsloth rät generell von Qwen3.5-QLoRA wegen Quantisierungsabweichungen ab. Dieses Notebook stoppt bei fehlendem VRAM, abweichenden Prüfsummen, nicht verifiziertem Datensatz oder ungültigen Trainingsverlusten. Keine Qualitätsversprechen und kein automatischer Merge/Deployment. Trainiert ausschließlich den LoRA-Adapter, **niemals das vollständige Modell**.

**Wichtig für den Datenschutz:** Die synthetischen v3-Trainingsdaten und das öffentlich herunterladbare Basis-Modell werden auf Google-Server übertragen. Nicht mit privaten Dokumenten, Schlüsseln, persönlichen Daten oder geheimen Trainingsbeispielen verwenden. Colab kann die Sitzung jederzeit beenden. Optional kann ein Zwischenadapter nach Google Drive gespeichert werden; dies ist **kein** vollständiger Optimizer-Resume-Checkpoint.

**Identität:** `Qwen/Qwen3.5-9B` (kein `-Base`, keine abweichende Architektur). Native Qwen3.5-Chat-Vorlage, `enable_thinking=False`, exakt assistant-only maskierte Labels wie in `scripts/neral_qlora_train.py`, Rank 16, Alpha 32. Das Verfahren bleibt ein *Versuch*, bis ein echter T4-Lauf erfolgreich nachgewiesen ist.

In [ ]:
# 1 · Abhängigkeiten in der Colab-Umgebung installieren (keine lokale GPU)
import sys, subprocess
packages = [
    "transformers>=5.0,<6", "peft>=0.17,<1", "accelerate>=1.5,<2",
    "bitsandbytes>=0.46,<1", "safetensors>=0.5,<1",
    "packaging>=24,<30", "sentencepiece>=0.2,<1",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Abhängigkeiten installiert. Bei Inkompatibilitäten hier abbrechen, nicht unbemerkt downgraden.")

In [ ]:
# 2 · T4- und Präzisionsprüfung
import torch, os, math, json, hashlib, time
from pathlib import Path
from collections.abc import Mapping
assert torch.cuda.is_available(), "Keine GPU verfügbar: Laufzeittyp T4 GPU wählen."
assert torch.cuda.device_count()==1, "Genau eine GPU erwartet."
name = torch.cuda.get_device_name(0)
free, total = torch.cuda.mem_get_info()
print("GPU:",name, "| Gesamt VRAM (GiB):",round(total/1024**3,2),"| Frei:",round(free/1024**3,2))
assert total >= 14*1024**3, "Für dieses 9B-Experiment mindestens ca. 14 GiB VRAM erforderlich."
assert free >= 12*1024**3, "GPU-Speicher bereits belegt. Neue T4-Laufzeit benutzen."
assert not torch.cuda.is_bf16_supported() or "T4" not in name, "GPU/Treiber-Angabe widersprüchlich."
MODEL_ID = "Qwen/Qwen3.5-9B"
MAX_LENGTH = 1024
RANK = 16
ALPHA = 32
EPOCHS = 1  # konservativer Start; KEINE automatische Änderung bei Fehlern
SEED = 20261117
SAVE_TO_DRIVE = False  # bei True: Google-Drive-Anmeldung und Adapter-Snapshots
SAVE_EVERY_STEPS = 20
torch.manual_seed(SEED)
print("T4 FP16 QLoRA-Protokoll; BF16 ausdrücklich deaktiviert.")

In [ ]:
# 3 · Ein geprüftes GitHub-Actions-Artefakt (ZIP) hochladen
from google.colab import files
import io, zipfile, shutil
uploaded = files.upload()
assert len(uploaded)==1, "Exakt eine ZIP-Datei mit vier verifizierten Datensatzdateien hochladen."
name = next(iter(uploaded))
assert name.lower().endswith(".zip"), "Bitte das unveränderte GitHub-Actions-ZIP hochladen."
DATA_ROOT = Path("/content/neral_v3_data")
assert not DATA_ROOT.exists(), "Für einen erneuten Upload: Laufzeit neu starten, keine Dateien überschreiben."
DATA_ROOT.mkdir()
with zipfile.ZipFile(io.BytesIO(uploaded[name])) as archive:
    members = archive.infolist()
    assert len(members) <= 40
    assert sum(m.file_size for m in members) <= 20_000_000, "Datensatz-ZIP zu groß."
    for m in members:
        p = Path(m.filename)
        assert not p.is_absolute() and ".." not in p.parts, "Unsicherer ZIP-Pfad."
        assert not (m.external_attr >> 16) & 0o170000 == 0o120000, "Symlink in ZIP untersagt."
    archive.extractall(DATA_ROOT)
def only(name):
    matches=list(DATA_ROOT.rglob(name))
    assert len(matches)==1 and matches[0].is_file(), f"Erwartete Datei {name} fehlt oder ist mehrfach enthalten."
    return matches[0]
TRAIN_FILE=only("train.jsonl")
VAL_FILE=only("validation.jsonl")
META_FILE=only("manifest.json")
PROOF_FILE=only("docker_proof.json")
print("Datensatz-ZIP eingelesen; prüfe jetzt Provenienz und Hashes.")

In [ ]:
# 4 · Fail-closed Provenienz, Split-Isolation und unabhängigen Docker-Beleg prüfen
sha256 = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
meta=json.loads(META_FILE.read_text("utf8"))
proof=json.loads(PROOF_FILE.read_text("utf8"))
assert meta["base_model"]==MODEL_ID and meta["expected_train"]==480 and meta["expected_validation"]==60
assert meta["training_performed"] is False
assert meta["exclusion"]["overlap_count"]==0
assert meta["exclusion"]["train_val_overlap_count"]==0
assert meta["exclusion"]["protected_answers_used_for_training"] is False
assert proof["status"]=="v3_all_oracles_verified_linux_docker"
assert proof["verified_rows"]==540 and proof["tool_rows"]==448 and proof["style_rows"]==92
assert proof["no_training"] is True and proof["no_gpu"] is True
for key, path in [("train",TRAIN_FILE),("validation",VAL_FILE)]:
    assert sha256(path)==meta["split_sha256"][key]==proof[key+"_sha256"], "Datensatz-Bytes verändert!"
def read_rows(path, n):
    rows=[json.loads(x) for x in path.read_text("utf8").splitlines() if x.strip()]
    assert len(rows)==n
    for row in rows:
        assert [m.get("role") for m in row["messages"]]==["system","user","assistant"]
        question=row["messages"][1]["content"]
        fingerprint=hashlib.sha256(" ".join(question.casefold().split()).encode("utf8")).hexdigest()
        assert row["id"]==fingerprint
    return rows
train_rows=read_rows(TRAIN_FILE,480)
val_rows=read_rows(VAL_FILE,60)
assert len({r["id"] for r in train_rows+val_rows})==540, "Train/Validation-Überlappung"
print("✅ 480 Trainings- und 60 Validierungsbeispiele; alle Hashes stimmen mit Linux-Docker-Proof überein.")
print("Provenienz:",meta["dataset_id"],"| Synthetic not human reviewed:",meta["synthetic_not_human_reviewed"])

In [ ]:
# 5 · Optional Google Drive für Zwischenadapter nutzen
from datetime import datetime, timezone
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LOCAL_OUT = Path("/content/neral_helios_v3_" + RUN_ID)
assert not LOCAL_OUT.exists()
LOCAL_OUT.mkdir(parents=True)
DRIVE_OUT=None
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUT=Path("/content/drive/MyDrive/NERAL_Colab_Adapters") / RUN_ID
    assert not DRIVE_OUT.exists(), "Nicht über einen alten Trainingslauf schreiben."
    DRIVE_OUT.mkdir(parents=True)
print("Ausgabe:", LOCAL_OUT, "| Drive-Snapshots:", bool(DRIVE_OUT))

In [ ]:
# 6 · Dasselbe native Qwen3.5-Chat-Template und assistant-only-Masking wie lokal
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,use_fast=True)
assert tokenizer.pad_token_id is not None, "Pad-Token fehlt"
def encode_answer_only(rows,tokenizer,max_seq_length):
    examples=[]
    for row in rows:
        turns=row["messages"]
        prefix=tokenizer.apply_chat_template(turns[:-1],tokenize=True,
                   add_generation_prompt=True,enable_thinking=False)
        complete=tokenizer.apply_chat_template(turns,tokenize=True,
                   add_generation_prompt=False,enable_thinking=False)
        prefix=prefix.get("input_ids") if isinstance(prefix,Mapping) else prefix
        complete=complete.get("input_ids") if isinstance(complete,Mapping) else complete
        if not isinstance(prefix,list) or not isinstance(complete,list) or complete[:len(prefix)]!=prefix:
            raise RuntimeError("Native Qwen3.5-Chat-Template-Inkompatibilität")
        if len(complete)>max_seq_length or len(complete)<=len(prefix):
            raise RuntimeError(f"Nicht still kürzen: {len(complete)} Tokens überschreiten {max_seq_length}")
        labels=[-100]*len(prefix)+complete[len(prefix):]
        assert any(x>=0 for x in labels)
        examples.append({"input_ids":complete,"attention_mask":[1]*len(complete),"labels":labels})
    return examples
train_tokens=encode_answer_only(train_rows,tokenizer,MAX_LENGTH)
val_tokens=encode_answer_only(val_rows,tokenizer,MAX_LENGTH)
print("Native Template geprüft; größte Länge:",max(map(lambda r:len(r["input_ids"]),train_tokens+val_tokens)))
class FixedTokenDataset:
    def __init__(self,examples):self.examples=examples
    def __len__(self):return len(self.examples)
    def __getitem__(self,i):return self.examples[i]
def collate(batch):
    width=max(len(r["input_ids"]) for r in batch)
    return {field:torch.tensor([
        row[field]+([ -100 if field=="labels" else (tokenizer.pad_token_id if field=="input_ids" else 0) ]*(width-len(row[field])))
        for row in batch],dtype=torch.long)
        for field in ("input_ids","attention_mask","labels")}

In [ ]:
# 7 · FP16 4-Bit NF4 Qwen3.5-9B + Rank16 PEFT, fail-closed T4-Speicherprüfung
import gc
from transformers import Qwen3_5ForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
torch.cuda.empty_cache()
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
base=Qwen3_5ForCausalLM.from_pretrained(MODEL_ID,quantization_config=quant,
    dtype=torch.float16,device_map={"":0},low_cpu_mem_usage=True)
base.config.use_cache=False
# Identisch mit dem lokalen speicherbegrenzten PEFT-Ansatz:
staged=[]
seen=set()
for module in (base.get_input_embeddings(),base.get_output_embeddings()):
    if module is not None and id(module) not in seen and module.weight.numel()>=100_000_000:
        seen.add(id(module))
        staged.append((module,module.weight.device,module.weight.dtype))
        module.to("cpu")
base=prepare_model_for_kbit_training(base,use_gradient_checkpointing=True,
                                     gradient_checkpointing_kwargs={"use_reentrant":False})
for module,dev,dtype in staged:
    module.to(device=dev,dtype=dtype)
    assert not module.weight.requires_grad
model=get_peft_model(base,LoraConfig(task_type="CAUSAL_LM",inference_mode=False,
    r=RANK,lora_alpha=ALPHA,lora_dropout=0,target_modules="all-linear",bias="none"))
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
assert 0<trainable<180_000_000, "Unerwartete trainierbare Parameter"
free,total=torch.cuda.mem_get_info()
print("LoRA trainierbare Parameter:",trainable,"| GPU frei nach Laden:",round(free/1024**3,2),"GiB")
assert free>1.4*1024**3, "Nach Laden zu wenig VRAM für Training auf T4. Kein GPU-Training starten."

In [ ]:
# 8 · Lokaler Originalverlust: assistant-only shifted CE über beaufsichtigte Positionen
import torch.nn.functional as F
from transformers import Trainer, TrainingArguments, TrainerCallback
def answer_only_loss(model,inputs,num_items_in_batch=None):
    tensors=dict(inputs)
    labels=tensors.pop("labels")
    assert labels.ndim==2 and labels.shape[0]==1
    positions=torch.nonzero(labels[0,1:]!=-100,as_tuple=True)[0]
    assert positions.numel()>0
    targets=labels[:,1:].index_select(1,positions)
    outputs=model(**tensors,logits_to_keep=positions,use_cache=False)
    logits=outputs.logits.float()
    loss=F.cross_entropy(logits.reshape(-1,logits.shape[-1]),targets.reshape(-1),reduction="sum")
    return loss/(num_items_in_batch if num_items_in_batch is not None else targets.numel()),outputs
class HeliosTrainer(Trainer):
    def __init__(self,**kwargs):
        super().__init__(**kwargs)
        self.model_accepts_loss_kwargs=True
    def compute_loss(self,model,inputs,return_outputs=False,num_items_in_batch=None):
        loss,out=answer_only_loss(model,inputs,num_items_in_batch)
        if not torch.isfinite(loss).item():
            raise FloatingPointError("Nicht-finite FP16-Loss: Qwen3.5-QLoRA auf T4 nicht stabil; Abbruch.")
        return (loss,out) if return_outputs else loss
class SafetyAndAdapterBackup(TrainerCallback):
    def on_log(self,args,state,control,logs=None,**kwargs):
        if logs:
            for key in ("loss","eval_loss"):
                if key in logs and not math.isfinite(float(logs[key])):
                    raise FloatingPointError("Nicht-finite "+key+" auf T4.")
        return control
    def on_step_end(self,args,state,control,model=None,**kwargs):
        if SAVE_TO_DRIVE and DRIVE_OUT and state.global_step and state.global_step%SAVE_EVERY_STEPS==0:
            assert model is not None
            out=DRIVE_OUT/f"step_{state.global_step:04d}_adapter"
            assert not out.exists()
            model.save_pretrained(str(out),safe_serialization=True)
            tokenizer.save_pretrained(str(out))
            print("Zwischenadapter nach Google Drive gesichert:",out.name)
        return control

train_args=TrainingArguments(
    output_dir=str(LOCAL_OUT/"trainer"),num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,gradient_accumulation_steps=8,
    per_device_eval_batch_size=1,eval_strategy="epoch",
    learning_rate=5e-5,lr_scheduler_type="cosine",warmup_steps=2,
    weight_decay=0.01,optim="paged_adamw_8bit",
    fp16=True,bf16=False,max_grad_norm=0.3,
    gradient_checkpointing=True,gradient_checkpointing_kwargs={"use_reentrant":False},
    save_strategy="no",logging_steps=5,report_to="none",seed=SEED,
    remove_unused_columns=False)
trainer=HeliosTrainer(model=model,args=train_args,
    train_dataset=FixedTokenDataset(train_tokens),
    eval_dataset=FixedTokenDataset(val_tokens),
    data_collator=collate,callbacks=[SafetyAndAdapterBackup()])
torch.cuda.reset_peak_memory_stats()
train_start=time.perf_counter()
result=trainer.train()
elapsed=time.perf_counter()-train_start
assert math.isfinite(float(result.training_loss))
assert result.global_step==math.ceil(480/8)*EPOCHS, "Angeforderte Trainingsepoche nicht vollständig"
print("GPU train steps:",result.global_step,"| Dauer (s):",round(elapsed,1),
      "| Peak VRAM GiB:",round(torch.cuda.max_memory_reserved()/1024**3,2))

In [ ]:
# 9 · Adapter-ZIP mit Manifest speichern, optional nach Drive sichern, herunterladen
import zipfile
ADAPTER = LOCAL_OUT/"adapter"
assert not ADAPTER.exists()
model.save_pretrained(str(ADAPTER),safe_serialization=True)
tokenizer.save_pretrained(str(ADAPTER))
weights=ADAPTER/"adapter_model.safetensors"
assert weights.is_file() and (ADAPTER/"adapter_config.json").is_file()
metadata={
 "status":"adapter_saved_not_independently_evaluated",
 "base_model":MODEL_ID,"train_sha256":sha256(TRAIN_FILE),
 "validation_sha256":sha256(VAL_FILE),
 "original_linux_docker_proof_sha256":sha256(PROOF_FILE),
 "train_rows":480,"val_rows":60,"rank":16,"alpha":32,
 "assistant_only_labels":True,"native_qwen35_template":True,
 "thinking":False,"precision":"fp16","quantization":"4bit_nf4",
 "epochs":EPOCHS,"optimizer_steps":result.global_step,
 "loss":float(result.training_loss),"adapter_sha256":sha256(weights),
 "peak_gpu_reserved_gib":round(torch.cuda.max_memory_reserved()/1024**3,2),
 "no_accuracy_claim":True
}
(LOCAL_OUT/"training_manifest.json").write_text(json.dumps(metadata,indent=2)+"\n","utf8")
ZIP=LOCAL_OUT.with_suffix(".zip")
with zipfile.ZipFile(ZIP,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=5) as z:
    for p in ADAPTER.rglob("*"):
        if p.is_file():z.write(p,arcname=str(Path("adapter")/p.relative_to(ADAPTER)))
    z.write(LOCAL_OUT/"training_manifest.json",arcname="training_manifest.json")
assert ZIP.is_file() and ZIP.stat().st_size>5000
if SAVE_TO_DRIVE and DRIVE_OUT:
    saved=DRIVE_OUT/ZIP.name
    assert not saved.exists()
    shutil.copy2(ZIP,saved)
    print("Adapter-ZIP zusätzlich gesichert:",saved.name)
print("Adapter fertig (nur Training, noch keine Qualitätsprüfung). ZIP MiB:",round(ZIP.stat().st_size/1024**2,2))
files.download(str(ZIP))